# LoRA Training: Hamlet Roleplay in Plain English

This notebook fine-tunes a LoRA adapter so TinyLlama still answers *as Hamlet*, but in clear modern English. The raw play text is first normalized into a plain-English Hamlet corpus, and only then converted into chat-style roleplay examples for training.

> Important: the plain-English rewrites in this notebook are heuristic approximations. They are a preprocessing step for stylistic normalization, not the training objective itself.


## Imports, paths, and run configuration

This cell keeps the notebook runnable from either the repository root or `training/` by discovering the repo root dynamically. It also centralizes the preprocessing, dataset, and training settings so the later cells stay easy to audit.

Required packages: `torch`, `transformers`, `datasets`, `peft`, and `accelerate`.


In [ ]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Iterable

import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)


def find_repo_root() -> Path:
    """Locate the repository root from a notebook or script context."""
    search_roots: list[Path] = []
    if "__file__" in globals():
        search_roots.append(Path(__file__).resolve().parent)
    search_roots.append(Path.cwd())

    for start in search_roots:
        for candidate in (start, *start.parents):
            if (candidate / "data" / "hamlet_onlyhamletraw.txt").is_file():
                return candidate

    raise FileNotFoundError(
        "Could not find the repository root. Run the notebook from the repo root "
        "or from the training/ directory so data/hamlet_onlyhamletraw.txt is visible."
    )


REPO_ROOT = find_repo_root()
RAW_TEXT_PATH = REPO_ROOT / "data" / "hamlet_onlyhamletraw.txt"
OUTPUT_DIR = REPO_ROOT / "models" / "lora_hamlet_3"

BASE_MODEL_NAME = "LiquidAI/LFM2-8B-A1B"
MAX_LENGTH = 512
NUM_EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 2e-4
EVAL_SPLIT = 0.1
SEED = 42
MIN_WORDS_PER_SPEECH = 4

SYSTEM_PROMPT_ROLEPLAY = (
    "You are Hamlet, Prince of Denmark. Speak in clear modern English while "
    "preserving Hamlet's introspection, melancholy, philosophical wit, and "
    "moral tension. Stay in character."
)

if not RAW_TEXT_PATH.is_file():
    raise FileNotFoundError(f"Missing raw text file: {RAW_TEXT_PATH}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"Raw Hamlet text: {RAW_TEXT_PATH}")
print(f"LoRA output dir: {OUTPUT_DIR}")
print(f"CUDA available: {torch.cuda.is_available()}")


Repository root: /home/jj/repos/DSCI490-Shakespearean-Personality-LLM-Augmentation-
Raw Hamlet text: /home/jj/repos/DSCI490-Shakespearean-Personality-LLM-Augmentation-/data/hamlet_onlyhamletraw.txt
LoRA output dir: /home/jj/repos/DSCI490-Shakespearean-Personality-LLM-Augmentation-/models/lora_hamlet_3
CUDA available: True


## Load and extract Hamlet speeches from the raw play text

The source file is speaker-tagged text, so the parser needs to start collecting when a line begins with `Ham.` and keep appending indented continuation lines until the next speaker appears. Stage directions are removed, but the wording of the dialogue is otherwise preserved.


In [ ]:
SPEAKER_PREFIX_RE = re.compile(r"^\s*([A-Za-z][A-Za-z0-9'_-]*)\.\s*(.*)$")
STAGE_DIRECTION_RE = re.compile(r"\[[^\]]+\]")
WHITESPACE_RE = re.compile(r"\s+")


def _clean_text(text: str) -> str:
    """Normalize spacing while preserving Hamlet's wording."""
    text = STAGE_DIRECTION_RE.sub(" ", text)
    text = text.replace("\u2014", "-").replace("\u2019", "'")
    return WHITESPACE_RE.sub(" ", text).strip()


def extract_hamlet_speeches(lines: Iterable[str]) -> list[str]:
    """Extract multi-line speeches that belong only to Hamlet."""
    speeches: list[str] = []
    current_speech: list[str] = []
    collecting_hamlet = False

    for raw_line in lines:
        line = raw_line.rstrip("\n")
        speaker_match = SPEAKER_PREFIX_RE.match(line)

        if speaker_match:
            if collecting_hamlet and current_speech:
                speeches.append(_clean_text(" ".join(current_speech)))

            speaker = speaker_match.group(1).lower()
            first_text = _clean_text(speaker_match.group(2))
            collecting_hamlet = speaker.startswith("ham")
            current_speech = [first_text] if collecting_hamlet and first_text else []
            continue

        if collecting_hamlet:
            cleaned = _clean_text(line)
            if cleaned:
                current_speech.append(cleaned)

    if collecting_hamlet and current_speech:
        speeches.append(_clean_text(" ".join(current_speech)))

    return [speech for speech in speeches if len(speech.split()) >= MIN_WORDS_PER_SPEECH]


hamlet_speeches = extract_hamlet_speeches(
    RAW_TEXT_PATH.read_text(encoding="utf-8").splitlines()
)

if not hamlet_speeches:
    raise ValueError(
        "No Hamlet speeches were extracted. Check the input file format and parser logic."
    )

average_words = sum(len(speech.split()) for speech in hamlet_speeches) / len(hamlet_speeches)

print(f"Extracted Hamlet speeches: {len(hamlet_speeches)}")
print(f"Average speech length: {average_words:.1f} words")
print("Sample extracted speech:")
print(hamlet_speeches[0][:260] + "...")


Extracted Hamlet speeches: 306
Average speech length: 34.4 words
Sample extracted speech:
A little more than kin, and less than kind!...


## Translate the speeches into plain English before dataset construction

The raw text does not need to remain Shakespearean inside the final training set. This cell normalizes Hamlet's lines into plainer English first, so the later dataset builder can treat the translated speeches as the assistant's actual character voice.

The result is still weak supervision: the targets are rule-based approximations, not literary translations.


In [ ]:
TRANSLATION_RULES: list[tuple[str, str]] = [
    (r"(?<![A-Za-z])i\s+prithee(?![A-Za-z])", "please"),
    (r"(?<![A-Za-z])prithee(?![A-Za-z])", "please"),
    (r"(?<![A-Za-z])methinks(?![A-Za-z])", "I think"),
    (r"(?<![A-Za-z])wherefore(?![A-Za-z])", "why"),
    (r"(?<![A-Za-z])ere\s+yet(?![A-Za-z])", "before"),
    (r"(?<![A-Za-z])'tis(?![A-Za-z])", "it is"),
    (r"(?<![A-Za-z])'twas(?![A-Za-z])", "it was"),
    (r"(?<![A-Za-z])thou(?![A-Za-z])", "you"),
    (r"(?<![A-Za-z])thee(?![A-Za-z])", "you"),
    (r"(?<![A-Za-z])thy(?![A-Za-z])", "your"),
    (r"(?<![A-Za-z])thine(?![A-Za-z])", "yours"),
    (r"(?<![A-Za-z])art(?![A-Za-z])", "are"),
    (r"(?<![A-Za-z])dost(?![A-Za-z])", "do"),
    (r"(?<![A-Za-z])doth(?![A-Za-z])", "does"),
    (r"(?<![A-Za-z])hast(?![A-Za-z])", "have"),
    (r"(?<![A-Za-z])hath(?![A-Za-z])", "has"),
    (r"(?<![A-Za-z])wilt(?![A-Za-z])", "will"),
    (r"(?<![A-Za-z])shalt(?![A-Za-z])", "shall"),
    (r"(?<![A-Za-z])canst(?![A-Za-z])", "can"),
    (r"(?<![A-Za-z])couldst(?![A-Za-z])", "could"),
    (r"(?<![A-Za-z])wouldst(?![A-Za-z])", "would"),
    (r"(?<![A-Za-z])shouldst(?![A-Za-z])", "should"),
    (r"(?<![A-Za-z])mayst(?![A-Za-z])", "may"),
    (r"(?<![A-Za-z])ere(?![A-Za-z])", "before"),
    (r"(?<![A-Za-z])whilst(?![A-Za-z])", "while"),
    (r"(?<![A-Za-z])ne'er(?![A-Za-z])", "never"),
    (r"(?<![A-Za-z])o'er(?![A-Za-z])", "over"),
    (r"(?<![A-Za-z])e'en(?![A-Za-z])", "even"),
    (r"i'\s*th'", "in the"),
]


def _match_case(source_text: str, replacement: str) -> str:
    """Keep replacements readable when the source starts with a capital letter."""
    letters = [character for character in source_text if character.isalpha()]
    if letters and all(character.isupper() for character in letters):
        return replacement.upper()
    if letters and letters[0].isupper():
        return replacement[0].upper() + replacement[1:]
    return replacement


def replace_case_aware(text: str, pattern: str, replacement: str) -> str:
    compiled = re.compile(pattern, flags=re.IGNORECASE)
    return compiled.sub(
        lambda match: _match_case(match.group(0), replacement),
        text,
    )


def shakespeare_to_plain_english(text: str) -> str:
    """Apply transparent heuristic rewrites to approximate plain English."""
    normalized = text
    for pattern, replacement in TRANSLATION_RULES:
        normalized = replace_case_aware(normalized, pattern, replacement)

    normalized = re.sub(r"\s+([,.;:!?])", r"\1", normalized)
    normalized = WHITESPACE_RE.sub(" ", normalized).strip()
    return normalized


plain_english_speeches = [
    shakespeare_to_plain_english(speech) for speech in hamlet_speeches
]
changed_flags = [
    source.lower() != target.lower()
    for source, target in zip(hamlet_speeches, plain_english_speeches)
]

normalized_corpus = Dataset.from_dict(
    {
        "original_text": hamlet_speeches,
        "plain_english_text": plain_english_speeches,
        "changed": changed_flags,
    }
)
changed_examples = normalized_corpus.filter(lambda row: row["changed"])

print(f"Total translated speeches: {len(normalized_corpus)}")
print(f"Speeches changed by normalization: {len(changed_examples)}")
example_row = changed_examples[0] if len(changed_examples) > 0 else normalized_corpus[0]
print("Original speech:", example_row["original_text"][:200] + "...")
print("Plain English speech:", example_row["plain_english_text"][:200] + "...")


Filter: 100%|██████████| 306/306 [00:00<00:00, 132794.31 examples/s]

Total translated speeches: 306
Speeches changed by normalization: 83
Original speech: Not so, my lord. I am too much i' th' sun....
Plain English speech: Not so, my lord. I am too much in the sun....


## Prepare roleplay training pairs from the translated speeches

The raw play text does not come with user prompts, so this cell converts each translated speech into synthetic roleplay supervision. The prompt builder uses lightweight topic heuristics to make the user side look like normal Hamlet questions, while the assistant target remains Hamlet speaking in plain English.

The train/eval split happens at the speech level before prompt expansion so held-out evaluation examples do not share the same source speech with the training set.


In [ ]:
TOPIC_PROMPT_RULES: list[tuple[str, re.Pattern[str], list[str]]] = [
    (
        "father",
        re.compile(r"\b(father|ghost|murder|murdered|memory)\b", re.IGNORECASE),
        [
            "How does your father's death weigh on you?",
            "What rises in you when you remember your father?",
        ],
    ),
    (
        "claudius",
        re.compile(r"\b(claudius|uncle|king)\b", re.IGNORECASE),
        [
            "What do you think of Claudius?",
            "How do you see the king who now rules Denmark?",
        ],
    ),
    (
        "gertrude",
        re.compile(r"\b(mother|queen|woman|gertrude)\b", re.IGNORECASE),
        [
            "How do you feel about your mother?",
            "What troubles you when you think of the queen?",
        ],
    ),
    (
        "ophelia",
        re.compile(r"\b(ophelia|love|lover|beauty|nunnery)\b", re.IGNORECASE),
        [
            "How do you think about Ophelia?",
            "What place does love have in your mind right now?",
        ],
    ),
    (
        "revenge",
        re.compile(r"\b(revenge|act|action|conscience|delay|hesitate)\b", re.IGNORECASE),
        [
            "Why do you hesitate when action is demanded of you?",
            "What stands between you and revenge?",
        ],
    ),
    (
        "mortality",
        re.compile(r"\b(death|dead|die|dying|grave|skull|dust|sleep|dream)\b", re.IGNORECASE),
        [
            "How do you think about death and what may follow it?",
            "What do thoughts of mortality do to your mind?",
        ],
    ),
]

DEFAULT_PROMPTS = [
    "What is on your mind at this moment?",
    "Speak your thoughts plainly.",
    "How do you see the world around you right now?",
]


def choose_roleplay_prompt(text: str, index: int) -> tuple[str, str]:
    """Map a translated speech to a lightweight user prompt."""
    for topic, pattern, prompts in TOPIC_PROMPT_RULES:
        if pattern.search(text):
            return prompts[index % len(prompts)], topic
    return DEFAULT_PROMPTS[index % len(DEFAULT_PROMPTS)], "general"


def split_sentences(text: str) -> list[str]:
    sentences = [
        segment.strip()
        for segment in re.split(r"(?<=[.!?])\s+", text)
        if segment.strip()
    ]
    return sentences if sentences else [text.strip()]


def build_roleplay_examples(speech_rows: Dataset) -> list[dict[str, str]]:
    examples: list[dict[str, str]] = []

    for index, row in enumerate(speech_rows):
        speech = row["plain_english_text"]
        prompt, topic = choose_roleplay_prompt(speech, index)
        examples.append(
            {
                "instruction": prompt,
                "response": speech,
                "topic": topic,
                "source_kind": "full_speech",
            }
        )

        sentences = split_sentences(speech)
        if len(sentences) >= 2:
            opening = sentences[0]
            continuation = " ".join(sentences[1:]).strip()
            if len(continuation.split()) >= MIN_WORDS_PER_SPEECH:
                examples.append(
                    {
                        "instruction": (
                            "Continue this thought as Hamlet in clear modern English:\n"
                            f"{opening}"
                        ),
                        "response": continuation,
                        "topic": topic,
                        "source_kind": "continuation",
                    }
                )

    return examples


speech_corpus = normalized_corpus.shuffle(seed=SEED)
eval_size = max(1, int(round(len(speech_corpus) * EVAL_SPLIT)))
if eval_size >= len(speech_corpus):
    raise ValueError("Need at least two speeches to create a train/eval split.")

speech_split = speech_corpus.train_test_split(test_size=eval_size, seed=SEED)
train_speech_dataset = speech_split["train"]
eval_speech_dataset = speech_split["test"]

train_dataset = Dataset.from_list(build_roleplay_examples(train_speech_dataset))
eval_dataset = Dataset.from_list(build_roleplay_examples(eval_speech_dataset))

print(f"Training speeches: {len(train_speech_dataset)}")
print(f"Evaluation speeches: {len(eval_speech_dataset)}")
print(f"Training roleplay examples: {len(train_dataset)}")
print(f"Evaluation roleplay examples: {len(eval_dataset)}")
print("Example training pair:")
print("Prompt:", train_dataset[0]["instruction"])
print("Response:", train_dataset[0]["response"][:400] + "...")


Training speeches: 275
Evaluation speeches: 31
Training roleplay examples: 424
Evaluation roleplay examples: 50
Example training pair:
Prompt: How do you think about death and what may follow it?
Response: 'Swounds, show me what you't do. Woo't weep? woo't fight? woo't fast? woo't tear thyself? Woo't drink up esill? eat a crocodile? I'll do't. Do you come here to whine? To outface me with leaping in her grave? Be buried quick with her, and so will I. And if you prate of mountains, let them throw Millions of acres on us, till our ground, Singeing his pate against the burning zone, Make Ossa like a wa...


## Tokenize the roleplay dataset and attach a LoRA adapter

Each example is tokenized as a chat conversation, but the loss is masked so training only applies to the assistant response. That keeps the objective focused on Hamlet's answer style in plain English rather than teaching the model to reproduce the system and user turns.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    model_dtype = torch.bfloat16
elif torch.cuda.is_available():
    model_dtype = torch.float16
else:
    model_dtype = torch.float32

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=model_dtype,
)
model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


def format_roleplay_prompt(instruction: str) -> str:
    return (
        f"<|system|>\n{SYSTEM_PROMPT_ROLEPLAY}</s>\n"
        f"<|user|>\n{instruction}</s>\n"
        "<|assistant|>\n"
    )


def tokenize_supervised_example(example: dict[str, str]) -> dict[str, list[int]]:
    prompt_text = format_roleplay_prompt(example["instruction"])
    response_text = example["response"] + "</s>"

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(response_text, add_special_tokens=False)["input_ids"]

    available_response_tokens = MAX_LENGTH - len(prompt_ids)
    if available_response_tokens <= 0:
        raise ValueError("Prompt length exceeded MAX_LENGTH before adding the response.")

    response_ids = response_ids[:available_response_tokens]
    input_ids = prompt_ids + response_ids
    attention_mask = [1] * len(input_ids)
    labels = [-100] * len(prompt_ids) + response_ids

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


tokenized_train_dataset = train_dataset.map(
    tokenize_supervised_example,
    remove_columns=train_dataset.column_names,
)
tokenized_eval_dataset = eval_dataset.map(
    tokenize_supervised_example,
    remove_columns=eval_dataset.column_names,
)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)

print(f"Tokenized train rows: {len(tokenized_train_dataset)}")
print(f"Tokenized eval rows: {len(tokenized_eval_dataset)}")


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9332.54it/s]


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


Map: 100%|██████████| 50/50 [00:00<00:00, 2817.35 examples/s]

Tokenized train rows: 424
Tokenized eval rows: 50


## Train the adapter and save it

Training uses HuggingFace `Trainer` with an epoch-level evaluation pass over the held-out roleplay split. Saving the adapter and tokenizer to a dedicated output directory makes the final inference cell reproducible in a fresh session.


In [ ]:
bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    bf16=bf16_ok,
    fp16=torch.cuda.is_available() and not bf16_ok,
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    data_collator=data_collator,
)

train_result = trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

print("Training metrics:", train_result.metrics)
print(f"Saved LoRA adapter + tokenizer to: {OUTPUT_DIR}")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
10,4.293136
20,4.002512
30,3.728309
40,3.733379
50,3.407012
60,3.530727
70,3.306763
80,3.596890
90,3.576183
100,3.445478


Training metrics: {'train_runtime': 467.1142, 'train_samples_per_second': 2.723, 'train_steps_per_second': 0.34, 'total_flos': 1377979540439040.0, 'train_loss': 3.5675214011714145, 'epoch': 3.0}
Saved LoRA adapter + tokenizer to: /home/jj/repos/DSCI490-Shakespearean-Personality-LLM-Augmentation-/models/lora_hamlet_3


## Reload the adapter and test both held-out prompts and free-form questions

The first check compares model outputs against held-out synthetic prompts. The second asks normal Hamlet questions to confirm the adapter behaves like a roleplay model rather than a translation model.


In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=model_dtype,
)
inference_model = PeftModel.from_pretrained(base_model, str(OUTPUT_DIR))
inference_model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
inference_model.to(device)


def build_roleplay_prompt(question: str) -> str:
    return (
        f"<|system|>\n{SYSTEM_PROMPT_ROLEPLAY}</s>\n"
        f"<|user|>\n{question}</s>\n"
        "<|assistant|>\n"
    )


def ask_hamlet(question: str, max_new_tokens: int = 160) -> str:
    prompt = build_roleplay_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated_tokens = outputs[0][inputs["input_ids"].shape[1] :]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


sample_count = min(3, len(eval_dataset))
for index in range(sample_count):
    example = eval_dataset[index]
    print(f"\nHeld-out example {index + 1}")
    print("Prompt:", example["instruction"])
    print("Reference:", example["response"])
    print("Model output:", ask_hamlet(example["instruction"]))

freeform_questions = [
    "How do you feel about your father's death?",
    "What do you think of Claudius?",
    "Why do you hesitate when revenge lies before you?",
]

for question in freeform_questions:
    print(f"\nFree-form question: {question}")
    print("Hamlet:", ask_hamlet(question))


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8615.44it/s]
Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Held-out example 1
Prompt: How do you think about Ophelia?
Reference: He poisons him in the garden for's estate. His name's Gonzago. The story is extant, and written in very choice Italian. You shall see anon how the murtherer gets the love of Gonzago's wife.


Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model output: And will he be your father too?

Held-out example 2
Prompt: Continue this thought as Hamlet in clear modern English:
He poisons him in the garden for's estate.
Reference: His name's Gonzago. The story is extant, and written in very choice Italian. You shall see anon how the murtherer gets the love of Gonzago's wife.


Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Model output: Come, sir, come!

Free-form question: How do you feel about your father's death?


Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hamlet: I would have thee say it to the world; but this is not the time for that. Speak.

Free-form question: What do you think of Claudius?


Both `max_new_tokens` (=160) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hamlet: He is a man to beware; for he is the father of Polonius, and by his death may have his daughter married. This will make her mad.

Free-form question: Why do you hesitate when revenge lies before you?
Hamlet: Ay, good sir; I am come to do you the service.
